# W9 Demo 1 - Reinforcement Learning Basic Concepts

This notebook is based on the lectures on Reinforcement Learning. It introduces the agent-environment loop, policies, rewards, returns, value functions, Bellman evaluation, temporal-difference updates, and exploration.

The examples use a tiny corridor world so that every table and update can be inspected by hand.

## 1. Core vocabulary

In reinforcement learning, an agent repeatedly observes a state, chooses an action, receives a reward, and moves to a next state.

Key terms used in the lecture:

- State: the information available to the agent at one time step.
- Action: a choice the agent can make.
- Reward: immediate feedback from the environment.
- Policy: a rule for choosing actions in states.
- Return: discounted sum of future rewards.
- Value function: expected return from a state under a policy.
- Model-based learning: learn or use a transition/reward model, then plan.
- Model-free learning: learn values, action utilities, or policies directly from experience.



In [1]:
from collections import defaultdict  # Import defaultdict, có thể dùng để tạo dictionary với giá trị mặc định
import random  # Import thư viện random để mô phỏng hành động ngẫu nhiên

# Hệ số chiết khấu dùng trong các bài toán quyết định tuần tự
GAMMA = 0.90

# Các hành động có thể thực hiện trong môi trường hành lang
ACTIONS = ("left", "right")

# Các trạng thái trong môi trường, từ 0 đến 4
STATES = (0, 1, 2, 3, 4)

# Các trạng thái kết thúc:
# 0 là trạng thái kết thúc xấu, 4 là trạng thái kết thúc tốt
TERMINALS = {0, 4}

# Môi trường hành lang:
# 0 là trạng thái terminal xấu, nhận thưởng âm.
# 4 là trạng thái terminal tốt, nhận thưởng dương.
# Các trạng thái không phải terminal có chi phí bước đi nhỏ.
def transition_distribution(state, action):
    # Nếu đang ở trạng thái kết thúc, agent sẽ ở nguyên đó
    # và không nhận thêm phần thưởng nào
    if state in TERMINALS:
        return [(1.0, state, 0.0)]

    # Nếu hành động là "left" thì hướng dự định là -1,
    # nếu là "right" thì hướng dự định là +1
    intended = -1 if action == "left" else 1

    # Hướng ngược lại với hướng dự định
    opposite = -intended

    # Danh sách các kết quả có thể xảy ra sau khi thực hiện hành động
    outcomes = []

    # Agent đi đúng hướng với xác suất 0.80,
    # và bị trượt sang hướng ngược lại với xác suất 0.20
    for probability, move in [(0.80, intended), (0.20, opposite)]:
        # Tính trạng thái kế tiếp sau khi di chuyển
        # max và min giúp đảm bảo trạng thái luôn nằm trong khoảng từ 0 đến 4
        next_state = max(0, min(4, state + move))

        # Gán phần thưởng dựa trên trạng thái kế tiếp
        if next_state == 0:
            reward = -1.0  # Đi vào trạng thái xấu
        elif next_state == 4:
            reward = 1.0   # Đi vào trạng thái tốt
        else:
            reward = -0.02 # Chi phí nhỏ cho mỗi bước đi trong trạng thái thường

        # Lưu lại một kết quả có thể xảy ra:
        # gồm xác suất, trạng thái kế tiếp và phần thưởng
        outcomes.append((probability, next_state, reward))

    # Trả về phân phối xác suất của các kết quả có thể xảy ra
    return outcomes


def step(state, action, rng=random):
    # Sinh một số ngẫu nhiên trong khoảng [0, 1)
    # để chọn kết quả theo phân phối xác suất
    sample = rng.random()

    # Biến dùng để cộng dồn xác suất
    total = 0.0

    # Duyệt qua các kết quả có thể xảy ra
    for probability, next_state, reward in transition_distribution(state, action):
        # Cộng dồn xác suất
        total += probability

        # Nếu mẫu ngẫu nhiên nằm trong khoảng xác suất hiện tại,
        # trả về trạng thái kế tiếp và phần thưởng tương ứng
        if sample <= total:
            return next_state, reward

    # Trường hợp dự phòng do sai số số học dấu phẩy động:
    # trả về kết quả cuối cùng trong phân phối
    return transition_distribution(state, action)[-1][1:]


# In ra danh sách các trạng thái
print("States:", STATES)

# In ra danh sách các hành động
print("Actions:", ACTIONS)

# Minh họa phân phối chuyển trạng thái từ state 2 khi chọn hành động "right"
print("Example transition from state 2 with action 'right':")

# Duyệt và in từng khả năng chuyển trạng thái
for p, s2, r in transition_distribution(2, "right"):
    print(f"  P={p:.2f} -> next_state={s2}, reward={r:+.2f}")

States: (0, 1, 2, 3, 4)
Actions: ('left', 'right')
Example transition from state 2 with action 'right':
  P=0.80 -> next_state=3, reward=-0.02
  P=0.20 -> next_state=1, reward=-0.02


## 2. Policies and episodes

A policy tells the agent what to do. A deterministic policy returns one action per state. A stochastic policy returns probabilities over actions.



In [2]:
# Chính sách đơn giản: luôn chọn hành động đi sang phải
def policy_go_right(state):
    return "right"


# Chính sách ngẫu nhiên: chọn ngẫu nhiên một hành động trong ACTIONS
def policy_random(state, rng=random):
    return rng.choice(ACTIONS)


# Chạy một episode, tức là một chuỗi các bước từ trạng thái bắt đầu
# cho đến khi gặp trạng thái kết thúc hoặc đạt số bước tối đa
def run_episode(policy, start_state=2, seed=7, max_steps=20):
    # Tạo bộ sinh số ngẫu nhiên với seed cố định
    # để kết quả có thể tái lập được
    rng = random.Random(seed)

    # Khởi tạo trạng thái ban đầu
    state = start_state

    # Danh sách lưu lại toàn bộ quá trình episode
    episode = []

    # Lặp tối đa max_steps bước
    for t in range(max_steps):
        # Nếu đang ở trạng thái kết thúc thì dừng episode
        if state in TERMINALS:
            break

        # Một số policy nhận cả state và rng,
        # một số policy chỉ nhận state.
        # Vì vậy dùng try-except để hỗ trợ cả hai kiểu policy.
        try:
            action = policy(state, rng)
        except TypeError:
            action = policy(state)

        # Thực hiện hành động trong môi trường,
        # nhận về trạng thái kế tiếp và phần thưởng
        next_state, reward = step(state, action, rng)

        # Lưu lại thông tin của bước hiện tại:
        # trạng thái hiện tại, hành động, phần thưởng, trạng thái kế tiếp
        episode.append((state, action, reward, next_state))

        # Cập nhật trạng thái hiện tại sang trạng thái kế tiếp
        state = next_state

    # Trả về toàn bộ episode đã chạy
    return episode


# In episode ra màn hình theo từng bước thời gian
def print_episode(episode):
    # enumerate giúp lấy cả chỉ số thời gian t và dữ liệu từng bước
    for t, (state, action, reward, next_state) in enumerate(episode):
        # In thông tin bước t:
        # s là state hiện tại, a là action,
        # r là reward, s' là state kế tiếp
        print(f"t={t:02d}: s={state}, a={action:>5}, r={reward:+.2f}, s'={next_state}")


# Chạy và in episode khi agent dùng policy ngẫu nhiên
print("Episode under a random policy:")
print_episode(run_episode(policy_random, seed=3))


# Chạy và in episode khi agent luôn chọn đi sang phải
print("\nEpisode under a simple 'always go right' policy:")
print_episode(run_episode(policy_go_right, seed=3))

Episode under a random policy:
t=00: s=2, a= left, r=-0.02, s'=1
t=01: s=1, a= left, r=-1.00, s'=0

Episode under a simple 'always go right' policy:
t=00: s=2, a=right, r=-0.02, s'=3
t=01: s=3, a=right, r=+1.00, s'=4


## 3. Discounted return

The return from time 0 is

$G_0 = r_1 + \gamma r_2 + \gamma^2 r_3 + ...$

The discount factor gamma ($\gamma$) controls how much the agent cares about future rewards.



In [3]:
# Tính tổng phần thưởng có chiết khấu theo thời gian
def discounted_return(rewards, gamma=GAMMA):
    # Biến lưu tổng phần thưởng sau khi chiết khấu
    total = 0.0

    # Trọng số chiết khấu ban đầu là 1,
    # tương ứng với phần thưởng ở bước đầu tiên
    weight = 1.0

    # Duyệt qua từng phần thưởng trong episode
    for reward in rewards:
        # Cộng phần thưởng đã nhân với trọng số chiết khấu vào tổng
        total += weight * reward

        # Cập nhật trọng số cho bước tiếp theo:
        # càng về sau thì phần thưởng càng bị chiết khấu nhiều hơn
        weight *= gamma

    # Trả về tổng phần thưởng có chiết khấu
    return total


# Chạy thử hai policy:
# - random: chọn hành động ngẫu nhiên
# - go_right: luôn đi sang phải
for name, policy in [("random", policy_random), ("go_right", policy_go_right)]:
    # Chạy một episode với policy hiện tại và seed cố định
    episode = run_episode(policy, seed=4)

    # Trích xuất danh sách reward từ episode
    # Mỗi phần tử episode có dạng: (state, action, reward, next_state)
    rewards = [reward for _, _, reward, _ in episode]

    # In ra danh sách phần thưởng nhận được trong episode
    print(f"{name:>8} rewards: {rewards}")

    # In ra tổng phần thưởng có chiết khấu của episode
    print(f"{name:>8} discounted return: {discounted_return(rewards):+.3f}\n")

  random rewards: [-0.02, -0.02, -0.02, -0.02, -0.02, -1.0]
  random discounted return: -0.672

go_right rewards: [-0.02, 1.0]
go_right discounted return: +0.880



## 4. Value functions and Bellman evaluation

For a fixed policy $\pi$, the value function $V_\pi(s)$ is the expected return from state s if the agent follows $\pi$.

The Bellman equation connects each state value to the reward and value of successor states:

$$V(s) = \sum_{s'} P(s' \mid s, \pi(s)) \left[ R(s, \pi(s), s') + \gamma V(s') \right]$$



In [4]:
# Đánh giá giá trị của một policy bằng phương pháp lặp Bellman
def evaluate_policy(policy, gamma=GAMMA, tolerance=1e-10, max_iterations=10_000):
    # Khởi tạo giá trị V(s) của tất cả trạng thái bằng 0
    values = {state: 0.0 for state in STATES}

    # Lặp tối đa max_iterations lần để cập nhật giá trị
    for iteration in range(max_iterations):
        # delta dùng để đo mức thay đổi lớn nhất giữa hai lần cập nhật
        delta = 0.0

        # Tạo bản sao của values để lưu giá trị mới
        # Tránh cập nhật trực tiếp làm ảnh hưởng đến các phép tính trong cùng một vòng lặp
        new_values = values.copy()

        # Duyệt qua từng trạng thái trong môi trường
        for state in STATES:
            # Nếu là trạng thái kết thúc thì giá trị bằng 0
            # vì sau đó không còn phần thưởng tương lai nào nữa
            if state in TERMINALS:
                new_values[state] = 0.0
                continue

            # Lấy hành động mà policy chọn tại trạng thái hiện tại
            action = policy(state)

            # Biến lưu giá trị kỳ vọng theo công thức Bellman
            expected = 0.0

            # Duyệt qua tất cả kết quả có thể xảy ra khi thực hiện action
            for probability, next_state, reward in transition_distribution(state, action):
                # Công thức Bellman:
                # V(s) = tổng theo các trạng thái kế tiếp:
                # P(s'|s,a) * [reward + gamma * V(s')]
                expected += probability * (reward + gamma * values[next_state])

            # Cập nhật giá trị mới cho trạng thái hiện tại
            new_values[state] = expected

            # Cập nhật delta để kiểm tra mức độ hội tụ
            delta = max(delta, abs(new_values[state] - values[state]))

        # Sau khi quét qua tất cả trạng thái, cập nhật bảng giá trị
        values = new_values

        # Nếu thay đổi nhỏ hơn tolerance thì xem như đã hội tụ
        if delta < tolerance:
            return values, iteration + 1

    # Nếu chưa hội tụ sau max_iterations lần lặp,
    # trả về giá trị hiện tại và số vòng lặp tối đa
    return values, max_iterations


# Đánh giá policy luôn đi sang phải
values_right, iterations = evaluate_policy(policy_go_right)

# In số lần quét Bellman cần để hội tụ
print(f"Converged after {iterations} Bellman sweeps")

# In giá trị V(s) của từng trạng thái theo policy go_right
for state in STATES:
    print(f"V_go_right({state}) = {values_right[state]:+.3f}")

Converged after 36 Bellman sweeps
V_go_right(0) = +0.000
V_go_right(1) = +0.284
V_go_right(2) = +0.694
V_go_right(3) = +0.921
V_go_right(4) = +0.000


## 5. One-step greedy improvement

If a value function is available, the agent can look one step ahead and choose the action with the highest expected utility. This is the planning idea behind dynamic programming and ADP.



In [5]:
# Tính giá trị Q(s, a) dựa trên bảng giá trị V(s)
def q_from_value(state, action, values, gamma=GAMMA):
    # Q(s, a) = tổng kỳ vọng của:
    # reward nhận được ngay lập tức + gamma * giá trị của trạng thái kế tiếp
    return sum(
        probability * (reward + gamma * values[next_state])
        for probability, next_state, reward in transition_distribution(state, action)
    )


# Chọn hành động tốt nhất tại một trạng thái dựa trên bảng giá trị V(s)
def greedy_action_from_values(state, values):
    # Với mỗi action, tính Q(s, a)
    # Sau đó chọn action có Q-value lớn nhất
    return max(ACTIONS, key=lambda action: q_from_value(state, action, values))


# In ra các hành động greedy được suy ra từ V_go_right
print("Greedy actions derived from V_go_right:")

# Duyệt qua tất cả trạng thái trong môi trường
for state in STATES:
    # Nếu là trạng thái kết thúc thì không cần chọn hành động
    if state in TERMINALS:
        print(f"state {state}: terminal")
    else:
        # Tính điểm Q(s, a) cho từng hành động tại state hiện tại
        action_scores = {a: q_from_value(state, a, values_right) for a in ACTIONS}

        # In ra điểm của từng hành động và hành động greedy tốt nhất
        print(
            f"state {state}: scores={action_scores}, "
            f"greedy={greedy_action_from_values(state, values_right)}"
        )

Greedy actions derived from V_go_right:
state 0: terminal
state 1: scores={'left': -0.6790496760294027, 'right': 0.2838012958823895}, greedy=right
state 2: scores={'left': 0.3501079913183669, 'right': 0.694168466502761}, greedy=right
state 3: scores={'left': 0.6838012958823896, 'right': 0.9209503239705974}, greedy=right
state 4: terminal


## 6. Temporal-difference learning

Temporal-difference learning is model-free. Instead of learning $P(s' | s, a)$, it updates a value estimate directly from an observed transition.

The basic $TD(0)$ update is:

$$V(s) \leftarrow V(s) + \alpha [ r + \gamma V(s') - V(s) ]$$

The bracketed term is **the TD error**.



In [6]:
# Học giá trị V(s) của một policy bằng thuật toán TD Learning
def td_learning(policy, episodes=800, alpha=0.08, gamma=GAMMA, seed=11):
    # Tạo bộ sinh số ngẫu nhiên với seed cố định
    # để kết quả chạy có thể tái lập
    rng = random.Random(seed)

    # values lưu giá trị ước lượng V(s)
    # defaultdict(float) giúp trạng thái chưa xuất hiện có giá trị mặc định là 0.0
    values = defaultdict(float)

    # snapshots lưu lại một số mốc giá trị trong quá trình học
    snapshots = []

    # Lặp qua nhiều episode để dần cải thiện ước lượng V(s)
    for episode_index in range(episodes):
        # Mỗi episode bắt đầu từ trạng thái 2
        state = 2

        # Giới hạn tối đa 30 bước cho mỗi episode
        for _ in range(30):
            # Nếu đã đến trạng thái kết thúc thì dừng episode
            if state in TERMINALS:
                break

            # Chọn hành động theo policy hiện tại
            action = policy(state)

            # Thực hiện hành động trong môi trường,
            # nhận về trạng thái kế tiếp và phần thưởng
            next_state, reward = step(state, action, rng)

            # TD target:
            # reward hiện tại + giá trị chiết khấu của trạng thái kế tiếp
            td_target = reward + gamma * values[next_state]

            # TD error là độ lệch giữa target và giá trị hiện tại đang ước lượng
            td_error = td_target - values[state]

            # Cập nhật giá trị V(s) theo công thức TD(0):
            # V(s) <- V(s) + alpha * TD_error
            values[state] += alpha * td_error

            # Chuyển sang trạng thái kế tiếp
            state = next_state

        # Lưu snapshot ở một số episode quan trọng
        # để quan sát quá trình học thay đổi như thế nào
        if episode_index in {0, 1, 2, 9, 99, episodes - 1}:
            snapshots.append((episode_index + 1, dict(values)))

    # Trả về bảng giá trị đã học và các snapshot
    return values, snapshots


# Chạy TD Learning cho policy luôn đi sang phải
td_values, snapshots = td_learning(policy_go_right)

# In các snapshot của giá trị V(s) trong quá trình học
print("TD snapshots for the always-go-right policy:")

for episode_index, values in snapshots:
    # Tạo dictionary gọn hơn:
    # lấy giá trị của từng state, nếu chưa có thì dùng 0.0,
    # sau đó làm tròn đến 3 chữ số thập phân
    compact = {state: round(values.get(state, 0.0), 3) for state in STATES}

    # In giá trị ước lượng sau một số lượng episode nhất định
    print(f"after {episode_index:>3} episodes: {compact}")


# In kết quả đánh giá policy bằng Bellman để làm mục tiêu so sánh
print("\nBellman evaluation target:")

# Làm tròn giá trị Bellman evaluation của policy go_right
print({state: round(values_right[state], 3) for state in STATES})

TD snapshots for the always-go-right policy:
after   1 episodes: {0: 0.0, 1: 0.0, 2: -0.002, 3: 0.08, 4: 0.0}
after   2 episodes: {0: 0.0, 1: -0.002, 2: 0.001, 3: 0.154, 4: 0.0}
after   3 episodes: {0: 0.0, 1: -0.002, 2: 0.011, 3: 0.221, 4: 0.0}
after  10 episodes: {0: 0.0, 1: -0.082, 2: 0.134, 3: 0.51, 4: 0.0}
after 100 episodes: {0: 0.0, 1: -0.103, 2: 0.665, 3: 0.963, 4: 0.0}
after 800 episodes: {0: 0.0, 1: 0.086, 2: 0.624, 3: 0.907, 4: 0.0}

Bellman evaluation target:
{0: 0.0, 1: 0.284, 2: 0.694, 3: 0.921, 4: 0.0}


## 7. Exploration versus exploitation

A purely greedy agent can miss good actions if it never tries them. The lecture connects this to exploration functions and optimistic priors. A simple practical version is epsilon-greedy action selection: usually exploit the current best action, but sometimes explore randomly.



In [7]:
# Chọn hành động theo chiến lược epsilon-greedy
def epsilon_greedy_action(q_values, state, epsilon, rng):
    # Với xác suất epsilon, chọn hành động ngẫu nhiên
    # Đây là bước exploration: thử khám phá hành động khác
    if rng.random() < epsilon:
        return rng.choice(ACTIONS)

    # Với xác suất 1 - epsilon, chọn hành động có Q-value lớn nhất
    # Đây là bước exploitation: tận dụng hành động đang được đánh giá tốt nhất
    return max(ACTIONS, key=lambda action: q_values[(state, action)])


# Khởi tạo bảng Q-value với giá trị mặc định là 0.0
q_values = defaultdict(float)

# Gán thử Q-value cho hành động "left" tại state 2
q_values[(2, "left")] = 0.10

# Gán thử Q-value cho hành động "right" tại state 2
# Vì 0.15 > 0.10 nên "right" là hành động greedy tại state 2
q_values[(2, "right")] = 0.15


# Tạo bộ sinh số ngẫu nhiên với seed cố định
# để kết quả có thể tái lập
rng = random.Random(21)

# Thử nhiều giá trị epsilon khác nhau
for epsilon in [0.0, 0.1, 0.5, 1.0]:
    # counts dùng để đếm số lần mỗi hành động được chọn
    counts = defaultdict(int)

    # Lặp 1000 lần để quan sát tần suất chọn hành động
    for _ in range(1000):
        # Chọn hành động bằng epsilon-greedy
        # rồi tăng bộ đếm cho hành động đó
        counts[epsilon_greedy_action(q_values, 2, epsilon, rng)] += 1

    # In kết quả đếm số lần chọn từng hành động
    # epsilon càng cao thì hành động ngẫu nhiên càng xuất hiện nhiều
    print(f"epsilon={epsilon:.1f}: {dict(counts)}")

epsilon=0.0: {'right': 1000}
epsilon=0.1: {'right': 955, 'left': 45}
epsilon=0.5: {'right': 754, 'left': 246}
epsilon=1.0: {'left': 496, 'right': 504}


## Takeaways

- A policy maps states to actions.
- Rewards are immediate; returns are accumulated and discounted.
- Value functions summarize expected future reward.
- Bellman equations connect neighboring state values.
- TD learning uses observed transitions instead of an explicit model.
- Exploration is necessary because early estimates can be wrong.

